# Explore the Norwegian Continental Shelf with KGLite Visual

This notebook builds the public SODIR knowledge graph with `kglite-datasets`, opens the saved `.kgl` file in KGLite Visual, and follows a bounded geological journey through fields, discoveries, wells, formation tops, physical evidence, and monthly production.

Every graph query has an explicit limit. The app remains open after **Run All** so you can continue exploring. The final cleanup cells are optional.

**Source meaning.** Production values are monthly aggregates from the [SODIR field production table](https://factpages.sodir.no/en/field/TableView/Production/Saleable/TotalNcsMonth). Formation-top depth follows the [SODIR wellbore attributes](https://factpages.sodir.no/en/wellbore/PageView/Exploration/All/6374) and is shown as measured depth in metres from Kelly bushing (MD m RKB). This notebook does not infer daily telemetry, well trajectories, or reservoir connectivity.

## One-time environment setup

You need Python 3.12 (the tested version) and Git for the pinned datasets checkout. Create an isolated environment first so the install does not depend on—or try to modify—your system Python. On Windows PowerShell, create it with `py -3.12 -m venv .venv` and activate it with `.\.venv\Scripts\Activate.ps1` instead of the first two commands below.

Install the released viewer and the exact engine and datasets inputs used by this notebook:

```bash
python3.12 -m venv .venv
source .venv/bin/activate
python -m pip install 'kglite-visual==0.1.8' 'kglite==0.17.3' matplotlib jupyter
python -m pip install 'kglite-datasets @ git+https://github.com/kkollsga/kglite-datasets.git@95794a2879143f305511e60db315cc61a1c319bd'
python -m ipykernel install --user --name kglite-sodir --display-name 'KGLite SODIR'
jupyter lab
```

Download `sodir-geologist.ipynb` from the guide into the directory where you created `.venv`, then launch `jupyter lab` from that directory as shown above. Open the downloaded notebook and select the **KGLite SODIR** kernel.

The source cache can be hundreds of megabytes; the validated 2026-09-08 graph was about 115 MB, and upstream refreshes can change its size and counts. Set `SODIR_PROJECT_DIR` before starting Jupyter to put them somewhere else. The default is `.kglite-sodir-notebook` beside this notebook. Cached source files are reused according to `kglite-datasets` cooldowns. The generated graph is reused only when its build record names the pinned datasets revision and its required geological and production capabilities pass a live query; otherwise the notebook rebuilds it. `SODIR_FORCE_REBUILD=1` still forces a rebuild.

In [ ]:
from __future__ import annotations

import calendar
from datetime import date, datetime
import gc
import http.client
import html as html_lib
import importlib.metadata
import json
import math
import os
from pathlib import Path
from urllib.parse import urljoin

from IPython.display import HTML, Image, display
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import kglite
import kglite_visual as kv

KGLITE_VERSION = "0.17.3"
DATASETS_VERSION = "0.1.16"
DATASETS_REVISION = "95794a2879143f305511e60db315cc61a1c319bd"
VIEWER_VERSION = "0.1.8"
PROTOCOL_VERSION = 10

assert importlib.metadata.version("kglite") == KGLITE_VERSION
assert importlib.metadata.version("kglite-datasets") == DATASETS_VERSION
assert importlib.metadata.version("kglite-visual") == VIEWER_VERSION

PROJECT_DIR = Path(os.environ.get("SODIR_PROJECT_DIR", ".kglite-sodir-notebook")).expanduser().resolve()
SOURCE_CACHE = PROJECT_DIR / "source-cache"
GRAPH_DIR = PROJECT_DIR / "graph"
EXPORT_DIR = PROJECT_DIR / "exports"
FIGURE_DIR = PROJECT_DIR / "figures"
GRAPH_PATH = GRAPH_DIR / "sodir.kgl"
PRODUCTION_PATH = GRAPH_DIR / "production-2000-latest.json"
GRAPH_BUILD_PATH = GRAPH_DIR / "sodir-build.json"
for directory in (SOURCE_CACHE, GRAPH_DIR, EXPORT_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print(f"Project: {PROJECT_DIR.name}/")


## Build once, reuse deliberately

The loader owns `source-cache/`. A normal rerun reuses both that cache and the portable graph. `SODIR_FORCE_REBUILD=1` asks the loader to reconsider its cached inputs under its 14-day index and 30-day dataset cooldowns. While the builder graph is open, the next cell uses type-aware Cypher `ts_series` expressions to extract only three fields and the oil channel from January 2000 through the current year into a bounded provenance sidecar. It then releases the builder before the file-backed viewer starts. KGLite 0.17.3 still records a false static schema warning for these typed time-series pseudo-properties even though it returns their values. This cell temporarily silences only the warning announcement, inspects the structured warnings, rejects anything unexpected, and retains the exact length, date, unit, and value checks.

In [ ]:
FIELDS = ["GULLFAKS", "OSEBERG", "DRAUGEN"]
PRODUCTION_START_YEAR = 2000
PRODUCTION_END_YEAR = date.today().year
assert 2 <= len(FIELDS) <= 3
assert 1960 <= PRODUCTION_START_YEAR <= PRODUCTION_END_YEAR <= 2100


def serial_value(value):
    if value is None:
        return None
    value = float(value)
    return value if math.isfinite(value) else None


def extract_production(graph):
    config = graph.timeseries_config("ProductionProfile")
    assert config["resolution"] == "month"
    assert config["units"]["prd_oil_net"] == "MillSm3"
    production = {}
    query = (
        "MATCH (p:ProductionProfile)-[:OF_FIELD]->(f:Field) "
        "WHERE f.title = $field "
        "RETURN ts_series(p.prd_oil_net, '{start}', '{end}') AS oil LIMIT 1"
    ).format(start=PRODUCTION_START_YEAR, end=PRODUCTION_END_YEAR)
    for field in FIELDS:
        warning_policy = kglite.get_query_warning_policy()
        kglite.set_query_warning_policy("silent")
        try:
            result = graph.cypher(query, params={"field": field})
        finally:
            kglite.set_query_warning_policy(warning_policy)
        unexpected = [message for message in result.warnings
                      if not ("RETURN projects property" in message
                              and "ProductionProfile node has" in message
                              and "prd_oil_net" in message)]
        if unexpected:
            raise RuntimeError(f"unexpected production-query warning: {unexpected}")
        row = result.to_dicts()[0]
        def by_month(items):
            values = {}
            for item in items:
                month = str(item["time"])[:7]
                parsed = datetime.strptime(month, "%Y-%m").date()
                if not (PRODUCTION_START_YEAR <= parsed.year <= PRODUCTION_END_YEAR) or month in values:
                    raise ValueError(f"unexpected or duplicate production month: {month}")
                values[month] = serial_value(item["value"])
            return values
        oil = by_month(row["oil"])
        months = sorted(oil)
        if not months:
            raise RuntimeError(f"no production observations for {field}")
        production[field] = [
            {"month": month, "oil_million_sm3": oil.get(month)}
            for month in months
        ]
    return {
        "provenance": {"kglite": KGLITE_VERSION, "kglite_datasets": DATASETS_VERSION,
                       "kglite_datasets_revision": DATASETS_REVISION,
                       "start_year": PRODUCTION_START_YEAR,
                       "end_year": PRODUCTION_END_YEAR, "fields": FIELDS,
                       "source": "ProductionProfile packed monthly time series",
                       "resolution": config["resolution"],
                       "oil_unit": config["units"]["prd_oil_net"]},
        "series": production,
    }

REQUIRED_GRAPH_CAPABILITIES = {
    "fields": "MATCH (f:Field) RETURN count(f) AS count LIMIT 1",
    "discoveries": "MATCH (d:Discovery) RETURN count(d) AS count LIMIT 1",
    "production_profiles": "MATCH (p:ProductionProfile)-[:OF_FIELD]->(:Field) RETURN count(p) AS count LIMIT 1",
    "formation_tops": "MATCH (:Wellbore)-[top:HAS_FORMATION_TOP]->(:Stratigraphy) RETURN count(top) AS count LIMIT 1",
    "core_records": "MATCH (c:WellboreCore)-[:OF_WELLBORE]->(:Wellbore) RETURN count(c) AS count LIMIT 1",
    "dst_records": "MATCH (t:WellboreDST)-[:OF_WELLBORE]->(:Wellbore) RETURN count(t) AS count LIMIT 1",
}


def graph_capabilities(graph):
    return {name: int(graph.cypher(query).to_dicts()[0]["count"])
            for name, query in REQUIRED_GRAPH_CAPABILITIES.items()}


expected_build = {
    "kglite": KGLITE_VERSION,
    "kglite_datasets": DATASETS_VERSION,
    "kglite_datasets_revision": DATASETS_REVISION,
}
def graph_cache_is_reusable(build_record, expected, capabilities):
    return (isinstance(build_record, dict)
            and all(build_record.get(key) == value for key, value in expected.items())
            and isinstance(capabilities, dict)
            and set(capabilities) == set(REQUIRED_GRAPH_CAPABILITIES)
            and build_record.get("capabilities") == capabilities
            and all(isinstance(count, int) and count > 0
                    for count in capabilities.values()))


try:
    cached_build = json.loads(GRAPH_BUILD_PATH.read_text())
except (FileNotFoundError, json.JSONDecodeError):
    cached_build = None
force_rebuild = os.environ.get("SODIR_FORCE_REBUILD") == "1"
build_metadata_matches = (isinstance(cached_build, dict)
                          and all(cached_build.get(key) == value
                                  for key, value in expected_build.items()))
needs_graph = force_rebuild or not GRAPH_PATH.is_file() or not build_metadata_matches
if not needs_graph:
    cached_graph = kglite.load(str(GRAPH_PATH))
    try:
        cached_capabilities = graph_capabilities(cached_graph)
    finally:
        del cached_graph
    needs_graph = not graph_cache_is_reusable(
        cached_build, expected_build, cached_capabilities)

expected_series = {"kglite": KGLITE_VERSION, "kglite_datasets": DATASETS_VERSION,
                   "kglite_datasets_revision": DATASETS_REVISION,
                   "start_year": PRODUCTION_START_YEAR,
                   "end_year": PRODUCTION_END_YEAR, "fields": FIELDS,
                   "source": "ProductionProfile packed monthly time series",
                   "resolution": "month", "oil_unit": "MillSm3"}
try:
    cached_production = json.loads(PRODUCTION_PATH.read_text())
except (FileNotFoundError, json.JSONDecodeError):
    cached_production = None
needs_series = (force_rebuild or not isinstance(cached_production, dict)
                or cached_production.get("provenance") != expected_series)
if needs_graph:
    from kglite_datasets import sodir
    builder_graph = sodir.open(
        str(SOURCE_CACHE), storage="memory", index_cooldown_days=14,
        dataset_cooldown_days=30, use_complement=True, workers=10,
        force_rebuild=force_rebuild, verbose=True,
    )
    capabilities = graph_capabilities(builder_graph)
    if any(count <= 0 for count in capabilities.values()):
        raise RuntimeError(f"SODIR enhancement capabilities are incomplete: {capabilities}")
    production = extract_production(builder_graph)
    temporary_graph = GRAPH_PATH.with_suffix(".tmp.kgl")
    builder_graph.save(str(temporary_graph))
    os.replace(temporary_graph, GRAPH_PATH)
    build_record = {**expected_build, "capabilities": capabilities}
    temporary_build = GRAPH_BUILD_PATH.with_suffix(".tmp.json")
    temporary_build.write_text(json.dumps(build_record, indent=2) + "\n")
    os.replace(temporary_build, GRAPH_BUILD_PATH)
    del builder_graph
elif needs_series:
    builder_graph = kglite.load(str(GRAPH_PATH))
    production = extract_production(builder_graph)
    del builder_graph
else:
    production = cached_production

if needs_series or needs_graph:
    temporary_json = PRODUCTION_PATH.with_suffix(".tmp.json")
    temporary_json.write_text(json.dumps(production, indent=2) + "\n")
    os.replace(temporary_json, PRODUCTION_PATH)
gc.collect()

assert production["provenance"] == {
    "kglite": KGLITE_VERSION, "kglite_datasets": DATASETS_VERSION,
    "kglite_datasets_revision": DATASETS_REVISION,
    "start_year": PRODUCTION_START_YEAR, "end_year": PRODUCTION_END_YEAR,
    "fields": FIELDS,
    "source": "ProductionProfile packed monthly time series",
    "resolution": "month",
    "oil_unit": "MillSm3",
}
assert GRAPH_PATH.is_file() and GRAPH_PATH.stat().st_size > 0
print(f"Graph ready: {GRAPH_PATH.name} ({GRAPH_PATH.stat().st_size:,} bytes)")


## Start the file-backed workspace

Rerunning this cell closes only the viewer handle created by this notebook. The embedded app appears below in a local notebook; the link opens it in a separate tab so you can watch later cells update the same shared view. Remote kernels use `jupyter-server-proxy` when available or show an SSH forwarding hint.

In [ ]:
def request(method, route, body=None, *, raw=False, view=None):
    active_view = view or _sodir_view
    connection = http.client.HTTPConnection("127.0.0.1", active_view.port, timeout=45)
    try:
        payload = None if body is None else json.dumps(body)
        headers = {} if body is None else {"content-type": "application/json"}
        connection.request(method, route, payload, headers)
        response = connection.getresponse()
        data = response.read()
        if response.status >= 400:
            raise RuntimeError(f"{method} {route} -> {response.status}: {data[:500].decode(errors='replace')}")
        return (data, dict(response.getheaders())) if raw else json.loads(data)
    finally:
        connection.close()

previous_view = globals().get("_sodir_view")
if previous_view is not None and not previous_view.closed:
    previous_view.close()
os.environ["KGLITE_VISUAL_CONFIG_DIR"] = str(PROJECT_DIR / "saved-views")
_sodir_view = kv.show(str(GRAPH_PATH), open_browser=False, max_load_mb=1024,
                      name="SODIR NCS exploration", height=680)
try:
    session = request("GET", "/api/session")
    assert session["protocol_version"] == PROTOCOL_VERSION
    assert session["stats"]["node_count"] > 0 and session["stats"]["edge_count"] > 0
    print(f"Loaded {session['stats']['node_count']:,} nodes / {session['stats']['edge_count']:,} relations")
    request("POST", "/api/validate", {"query": "MATCH (f:Field) RETURN f LIMIT 1"})
except Exception:
    _sodir_view.close()
    raise RuntimeError(
        "This kernel does not have the required kglite-visual release. Repeat the one-time setup above."
    )
display(HTML(f'<p><a href="{_sodir_view.url}" target="_blank"><b>Open the live SODIR workspace in a new tab</b></a></p>'))
display(_sodir_view)


In [ ]:
def require_complete(result, label):
    meta = result.get("meta", result)
    for key in ("bound", "link_bound"):
        bound = meta.get(key)
        if bound and bound["truncated"]:
            raise RuntimeError(
                f"{label} was truncated: {bound['returned']} of {bound['total']}. "
                "Narrow the query before interpreting it."
            )
    return result


def table_rows(query, params=None, *, label, view=None, limit=100):
    result = request("POST", "/api/cypher", {
        "query": query, "params": params or {}, "limit": limit, "as_graph": False,
    }, view=view)
    require_complete(result, label)
    rows = [dict(zip(result["columns"], values)) for values in zip(*result["data"])]
    print(f"{label}: {result['bound']['returned']} of {result['bound']['total']} rows")
    return rows


def display_table(rows, title):
    if not rows:
        display(HTML(f"<p><b>{html_lib.escape(title)}:</b> no rows</p>"))
        return
    columns = list(rows[0])
    def cell(value):
        if value is None:
            return "<i>missing</i>"
        if isinstance(value, float):
            value = f"{value:,.2f}"
        return html_lib.escape(str(value))
    head = "".join(f"<th>{html_lib.escape(column)}</th>" for column in columns)
    body = "".join("<tr>" + "".join(f"<td>{cell(row.get(column))}</td>" for column in columns) + "</tr>" for row in rows)
    display(HTML(f"<h4>{html_lib.escape(title)}</h4><table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))


def capture_image(path, *, kernel="auto", width=1200, height=800):
    current = request("GET", "/api/view-state")
    settings = {"scope": "visible", "format": "png", "expected": current["stamp"],
                "subset_revision": current["subset_revision"], "kernel": kernel,
                "width": width, "height": height, "seed": 7, "theme": "light"}
    preview = request("POST", "/api/render/preview", settings)
    payload, _ = request("POST", "/api/render/download",
                         {**settings, "preview_digest": preview["preview"]["preview_digest"]},
                         raw=True)
    path.write_bytes(payload)
    print(f"Rendered {preview['preview']['nodes']} nodes / {preview['preview']['edges']} relations: {path.name}")
    display(Image(filename=str(path)))
    return preview


def show_graph(query, params=None, *, label, kernel=None, color_by=None, reset=True):
    if reset:
        request("POST", "/api/reset", {})
    result = request("POST", "/api/cypher", {
        "query": query, "params": params or {}, "limit": 100, "as_graph": True,
    })
    require_complete(result, label)
    if kernel:
        request("POST", "/api/layout", {"kernel": kernel})
    if color_by:
        current = request("GET", "/api/view-state")
        request("POST", "/api/appearance", {
            "color_by": color_by, "size_by": None, "expected": current["stamp"],
        })
    node_bound = result["meta"]["bound"]
    link_bound = result["meta"]["link_bound"]
    print(f"Explore — {label}: {node_bound['returned']} nodes, {link_bound['returned']} relations")
    return result


## 1. Where are recently discovered producing fields?

**Question.** How widely are the twelve most recently discovered coordinate-bearing producing fields distributed across the NCS?

The query loads twelve distinct fields whose source records carry geometry and switches to the geographic layout. Open **Data**, add `fldHcType`, `fldDiscoveryYear`, and `wkt_geometry`, then use **Appearance → Color by → fldHcType**. Read each plotted point as a representative location derived from source geometry; it is not a field-outline or reservoir-extent map.

In [ ]:
MAP_QUERY = """MATCH (f:Field)
WHERE f.fldCurrentActivitySatus = 'Producing' AND f.wkt_geometry IS NOT NULL
RETURN f
ORDER BY f.fldDiscoveryYear DESC
LIMIT 12"""
map_view = show_graph(MAP_QUERY, label="12 producing fields", kernel="geo", color_by="fldHcType")
map_render = capture_image(FIGURE_DIR / "producing-fields-map.png", kernel="geo")


## 2. How did early Johan Sverdrup appraisal progress?

**Question.** What chronology is visible in the first twelve completed wildcat and appraisal wellbores linked through the Johan Sverdrup discovery?

The count and ordered table report current source data. The view shows at most the earliest twelve linked wellbores, so it is a bounded early chronology rather than the full drilling history. The picture shows recorded graph relationships, not well trajectories. In **Data**, inspect purpose, content, measured depth, and final vertical depth; select a well to see its path through the discovery to the field.

In [ ]:
appraisal_count = table_rows(
    "MATCH (f:Field)<-[:IN_FIELD]-(d:Discovery)<-[:IN_DISCOVERY]-(w:Wellbore) "
    "WHERE f.title = $field AND w.wlbPurpose IN ['WILDCAT', 'APPRAISAL'] "
    "RETURN count(*) AS linked_wells LIMIT 1",
    {"field": "JOHAN SVERDRUP"}, label="Linked wildcat and appraisal count")
linked_well_count = appraisal_count[0]["linked_wells"]
assert linked_well_count > 0
print(f"The next view is the earliest {min(12, linked_well_count)} of {linked_well_count} linked wellbores.")

FIELD_CONTEXT_QUERY = """MATCH (f:Field)<-[rf:IN_FIELD]-(d:Discovery)<-[rd:IN_DISCOVERY]-(w:Wellbore)
WHERE f.title = $field AND w.wlbPurpose IN ['WILDCAT', 'APPRAISAL']
RETURN f,rf,d,rd,w
ORDER BY w.wlbCompletionDate
LIMIT 12"""
field_view = show_graph(FIELD_CONTEXT_QUERY, {"field": "JOHAN SVERDRUP"},
                        label="Johan Sverdrup discovery wells", kernel="radial", color_by="type")
appraisal_rows = table_rows(
    "MATCH (f:Field)<-[:IN_FIELD]-(d:Discovery)<-[:IN_DISCOVERY]-(w:Wellbore) "
    "WHERE f.title = $field AND w.wlbPurpose IN ['WILDCAT', 'APPRAISAL'] "
    "RETURN w.title AS wellbore, w.wlbPurpose AS purpose, "
    "w.wlbCompletionDate AS completed, w.wlbContent AS content "
    "ORDER BY completed LIMIT 12",
    {"field": "JOHAN SVERDRUP"}, label="Early appraisal chronology")
display_table(appraisal_rows, "Early appraisal chronology")
if appraisal_rows:
    print(f"Current bounded chronology: {appraisal_rows[0]['wellbore']} completed {appraisal_rows[0]['completed']} "
          f"through {appraisal_rows[-1]['wellbore']} completed {appraisal_rows[-1]['completed']}.")


## 3. What depth context is recorded for well 16/2-6?

**Question.** Which formation-level tops were reported along the discovery well, and at what measured depths?

The table is ordered by `lsuTopDepth`, measured as **MD m RKB**. It is not TVD and does not demonstrate reservoir connectivity. `lsuBottomDepth` is deliberately excluded because it is unavailable in the current source snapshot. Read the aliased depth in the executed **Query results · source** table below; it is an edge property and is not a node field. Select EKOFISK FM or TOR FM there, then use **Show selection in Explore** to relate that record back to the well.

In [ ]:
FORMATION_GRAPH_QUERY = """MATCH (w:Wellbore)-[top:HAS_FORMATION_TOP]->(s:Stratigraphy)
WHERE w.title = $well AND s.lsuLevel = 'FORMATION'
RETURN w,top,s
ORDER BY top.lsuTopDepth
LIMIT 20"""
formation_view = show_graph(FORMATION_GRAPH_QUERY, {"well": "16/2-6"},
                            label="16/2-6 formation tops", kernel="radial", color_by="type")
formation_rows = table_rows(
    "MATCH (w:Wellbore)-[top:HAS_FORMATION_TOP]->(s:Stratigraphy) "
    "WHERE w.title = $well AND s.lsuLevel = 'FORMATION' "
    "RETURN s.title AS formation, top.lsuTopDepth AS top_md_m_rkb "
    "ORDER BY top_md_m_rkb LIMIT 20",
    {"well": "16/2-6"}, label="Formation tops")
display_table(formation_rows, "Formation tops — MD m RKB")


### Add core and DST intervals

**Question.** Which core and drill-stem-test source records are linked to 16/2-6?

This bounded view contains three core records and one DST record. The figure places their numeric source depths beside three nearby formation-top markers. SODIR documents the formation tops as MD m RKB and the DST depths in metres; the core source says metres but does not explicitly establish the same datum. Treat overlap as a prompt for source review, not a formation assignment or connectivity claim. The reported DST oil rate belongs to that test record; it is not field production.

In [ ]:
COMBINED_EVIDENCE_QUERY = """MATCH (w:Wellbore)-[r]-(e)
WHERE w.title = $well
  AND ((type(r) = 'HAS_FORMATION_TOP' AND e.type = 'Stratigraphy' AND e.lsuLevel = 'FORMATION')
    OR (type(r) = 'OF_WELLBORE' AND e.type IN ['WellboreCore', 'WellboreDST']))
RETURN w,r,e,coalesce(r.lsuTopDepth,e.wlbCoreIntervalTop,e.wlbDstFromDepth) AS top_md
ORDER BY top_md
LIMIT 40"""
evidence_view = show_graph(COMBINED_EVIDENCE_QUERY, {"well": "16/2-6"},
                           label="16/2-6 tops, cores, and DST", kernel="radial",
                           color_by="type")
core_rows = table_rows(
    "MATCH (c:WellboreCore)-[:OF_WELLBORE]->(w:Wellbore) WHERE w.title = $well "
    "RETURN c.wlbCoreNumber AS core, c.wlbCoreIntervalTop AS top_md, "
    "c.wlbCoreIntervalBottom AS bottom_md, c.wlbCoreIntervalUom AS unit, "
    "c.wlbTotalCoreLength AS recovered_length ORDER BY top_md LIMIT 12",
    {"well": "16/2-6"}, label="Core records")
dst_rows = table_rows(
    "MATCH (t:WellboreDST)-[:OF_WELLBORE]->(w:Wellbore) WHERE w.title = $well "
    "RETURN t.wlbDstTestNumber AS test, t.wlbDstFromDepth AS from_md_m, "
    "t.wlbDstToDepth AS to_md_m, t.wlbDstOilProd AS oil_sm3_day, "
    "t.wlbDstGasProd AS gas_sm3_day ORDER BY from_md_m LIMIT 12",
    {"well": "16/2-6"}, label="DST records")
display_table(core_rows, "Core intervals")
display_table(dst_rows, "Drill-stem test")
selected_tops = {"INTRA DRAUPNE FM SS", "DRAUPNE FM", "SKAGERRAK FM"}
fig, axis = plt.subplots(figsize=(9, 6))
for row in formation_rows:
    if row["formation"] in selected_tops:
        depth = row["top_md_m_rkb"]
        axis.scatter(0, depth, s=55)
        axis.annotate(f"{row['formation']}  {depth:g}", (0, depth), xytext=(8, 0),
                      textcoords="offset points", va="center")
for row in dst_rows:
    axis.vlines(1, row["from_md_m"], row["to_md_m"], linewidth=10, color="#d97706")
    axis.annotate(f"DST {row['from_md_m']:g}–{row['to_md_m']:g}",
                  (1, (row["from_md_m"] + row["to_md_m"]) / 2), xytext=(8, 0),
                  textcoords="offset points", va="center")
for row in core_rows:
    axis.vlines(2, row["top_md"], row["bottom_md"], linewidth=10, color="#059669")
    axis.annotate(f"Core {row['core']}  {row['top_md']:g}–{row['bottom_md']:g}",
                  (2, (row["top_md"] + row["bottom_md"]) / 2), xytext=(8, 0),
                  textcoords="offset points", va="center")
axis.set_xticks([0, 1, 2], ["Formation tops\nMD m RKB", "DST interval\nMD m", "Core intervals\nmetres; datum unspecified"])
axis.set_ylabel("Reported source depth (m)")
axis.set_title("16/2-6 — numeric depth context (no formation assignment)")
axis.set_xlim(-0.4, 2.8)
axis.invert_yaxis()
axis.grid(axis="y", alpha=.25)
fig.tight_layout()
depth_chart_path = FIGURE_DIR / "depth-context-16-2-6.png"
fig.savefig(depth_chart_path, dpi=150)
plt.show()


## 4. How are fields connected to monthly production?

**Question.** Which production profiles are attached to the three comparison fields?

The graph stores monthly values as packed time series on `ProductionProfile`, not as month nodes. The app can show the fields and profile relationships; the next cells read only the bounded 2000-to-latest sidecar extracted through KGLite's type-aware Cypher `ts_series` expression.

In [ ]:
PROFILE_QUERY = """MATCH (p:ProductionProfile)-[r:OF_FIELD]->(f:Field)
WHERE f.title IN $fields
RETURN p,r,f
LIMIT 12"""
profile_view = show_graph(PROFILE_QUERY, {"fields": FIELDS},
                         label=f"{len(FIELDS)} fields and production profiles", kernel="islands", color_by="type")


### Compare production from 2000 through the latest available month

A monthly volume divided by that month's actual calendar days is labelled **monthly-average calendar-day rate**. Leap-year February therefore uses 29 days. Missing source values stay missing. These curves are derived from monthly aggregates and are not day-by-day telemetry. The end year follows the notebook run date; the chart stops at the latest month present in the source.


In [ ]:
def monthly_average_daily_rate(value, year, month, scale):
    if value is None:
        return None
    value = float(value)
    if not math.isfinite(value):
        return None
    return (value * scale) / calendar.monthrange(year, month)[1]


def oil_rate_series(field):
    values = []
    for row in production["series"][field]:
        year, month = map(int, row["month"].split("-"))
        values.append(monthly_average_daily_rate(row["oil_million_sm3"], year, month, 1_000_000))
    return values

months = [row["month"] for row in production["series"][FIELDS[0]]]
assert all([row["month"] for row in production["series"][field]] == months for field in FIELDS)
month_dates = [datetime.strptime(month, "%Y-%m") for month in months]
latest_month = months[-1]
fig, axis = plt.subplots(figsize=(11, 5.5))
for field in FIELDS:
    axis.plot(month_dates, oil_rate_series(field), linewidth=1.7, label=field)
axis.set_ylabel("Monthly-average oil rate (Sm³/day)")
axis.set_xlabel("Production month")
axis.xaxis.set_major_locator(mdates.YearLocator(5))
axis.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axis.grid(alpha=.25)
axis.legend(ncol=3)
axis.set_title(f"Gullfaks, Oseberg, and Draugen — {PRODUCTION_START_YEAR} to {latest_month}")
fig.tight_layout()
chart_path = FIGURE_DIR / "production-2000-latest.png"
fig.savefig(chart_path, dpi=150)
plt.show()

assert monthly_average_daily_rate(1, 2024, 2, 1_000_000) == 1_000_000 / 29
assert monthly_average_daily_rate(None, 2024, 2, 1_000_000) is None
oil_totals = {
    field: sum(row["oil_million_sm3"] for row in production["series"][field]
               if row["oil_million_sm3"] is not None)
    for field in FIELDS
}
display(HTML(
    f"<p><b>Source coverage:</b> {months[0]} through {latest_month}; "
    + html_lib.escape(", ".join(f"{field}: {oil_totals[field]:.3f} million Sm³ oil" for field in FIELDS))
    + ". Monthly-average rates are derived from these source totals.</p>"
))


### Build the same production comparison inside Visual

The earlier Python figure is an independent check. This cell validates a parameterized API query and prints an equivalent bounded literal query for the current text-only Query editor. Paste the printed query into **Query**, leave **Show in graph** unchecked, and select **Run**, then open **Data → Query results → Visualize result**. Choose **Line** and map `month` to x, `monthly_oil_million_sm3` to y, and `field` to series. Enable **monthly total → calendar-day average**, confirm the monthly contract, and use scale `1000000`. Set x label to **Production month**, y label to **Monthly-average oil rate**, and unit to **Sm³/day**. Suggested title: **Gullfaks, Oseberg, and Draugen — oil production since 2000**. The source table remains available beside the chart, and SVG/PNG export carries the query provenance.


In [ ]:
PRODUCTION_CHART_QUERY = """MATCH (p:ProductionProfile)-[:OF_FIELD]->(f:Field)
WHERE f.title IN $fields
UNWIND ts_series(p.prd_oil_net, '{start}', '{end}') AS observation
RETURN f.title AS field, observation.time AS month,
       observation.value AS monthly_oil_million_sm3
ORDER BY field, month
LIMIT 2000""".format(start=PRODUCTION_START_YEAR, end=PRODUCTION_END_YEAR)
production_chart_rows = table_rows(
    PRODUCTION_CHART_QUERY, {"fields": FIELDS}, label="monthly oil rows for Visual", limit=2000
)
production_counts = {
    field: sum(row["field"] == field for row in production_chart_rows)
    for field in FIELDS
}
assert set(production_counts) == set(FIELDS)
assert all(12 <= count <= 12 * (PRODUCTION_END_YEAR - PRODUCTION_START_YEAR + 1)
           for count in production_counts.values())
assert len(production_chart_rows) <= 2000
latest_chart_month = max(row["month"] for row in production_chart_rows)
PRODUCTION_VISUAL_QUERY = PRODUCTION_CHART_QUERY.replace("$fields", json.dumps(FIELDS))
display_table(production_chart_rows[:6], "First six validated monthly oil rows")
display(HTML(
    "<p><b>Selected-field coverage:</b> "
    + html_lib.escape(", ".join(f"{field}: {count} months" for field, count in production_counts.items()))
    + f"; latest source month: {html_lib.escape(str(latest_chart_month)[:7])}</p>"
      "<p>Paste this bounded query into <b>Query</b>, leave <b>Show in graph</b> unchecked, select <b>Run</b>, then "
      "<b>Visualize result</b>. Confirm the mappings and monthly-average transform described above, then select "
      "<b>Build chart</b>:</p>"
    + f'<pre style="white-space:pre-wrap">{html_lib.escape(PRODUCTION_VISUAL_QUERY)}</pre>'
))


## 5. Preserve a reviewable well-level handoff

Return to the specific 16/2-6 evidence assembled above: ordered formation tops, three cores, and one DST record. This is a reviewable source neighborhood rather than an unordered sample across hundreds of matching field patterns. Save it under a notebook-specific name, then preview and download its exact visible GraphML membership.

In [ ]:
handoff_view = show_graph(COMBINED_EVIDENCE_QUERY, {"well": "16/2-6"},
                          label="16/2-6 depth-context handoff", kernel="radial",
                          color_by="type")
current = request("GET", "/api/view-state")
saved = request("POST", "/api/views/save", {
    "name": "SODIR notebook — 16/2-6 depth context", "replace": True,
    "expected": current["stamp"],
})
current = request("GET", "/api/view-state")
preview_request = {
    "scope": "visible", "format": "graphml", "expected": current["stamp"],
    "subset_revision": current["subset_revision"],
}
preview = request("POST", "/api/export/preview", preview_request)
payload, headers = request("POST", "/api/export/download",
                           {**preview_request, "preview_digest": preview["preview_digest"]}, raw=True)
export_path = EXPORT_DIR / "well-16-2-6-depth-context.graphml"
export_path.write_bytes(payload)
print(f"Saved view and exported {preview['nodes']} nodes / "
      f"{preview['edges']} relations to {export_path.name}")
display(HTML(f'<a href="{_sodir_view.url}" target="_blank">Continue in the live workspace</a>'))


## Optional cleanup

**Run All stops above and leaves the workspace running.** Execute the first cell below only when you are finished. The second cell deletes notebook-generated exports and figures only after you change its guard. The downloaded SODIR cache and portable graph remain available for the next run.

In [ ]:
# Optional: stop only the viewer owned by this notebook.
# _sodir_view.close()


In [ ]:
# Optional: remove small notebook-generated artifacts, never the source cache or graph.
REMOVE_NOTEBOOK_OUTPUTS = False
if REMOVE_NOTEBOOK_OUTPUTS:
    import shutil
    shutil.rmtree(EXPORT_DIR, ignore_errors=True)
    shutil.rmtree(FIGURE_DIR, ignore_errors=True)
    print("Removed notebook-owned exports and figures.")
